In [ ]:
"""
FT-Transformer, bir Excel tablosundaki birbirinden farklı verileri (yaş, maaş, şehir gibi) alıp hepsini kendi anlayabileceği standart "bilgi paketlerine" dönüştürerek
işe başlar. Daha sonra, elindeki bu bilgilerin tamamına aynı anda bakıp aralarındaki ilişkileri ve birbirlerini nasıl etkilediklerini inceleyerek büyük resmi görür ve
sonuca ulaşır.
"""

'\nFT-Transformer, bir Excel tablosundaki birbirinden farklı verileri (yaş, maaş, şehir gibi) alıp hepsini kendi anlayabileceği standart "bilgi paketlerine" dönüştürerek \nişe başlar. Daha sonra, elindeki bu bilgilerin tamamına aynı anda bakıp aralarındaki ilişkileri ve birbirlerini nasıl etkilediklerini inceleyerek büyük resmi görür ve \nsonuca ulaşır.\n'

In [1]:
!pip install pytorch-tabular torch pandas scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.8/165.8 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 23.2 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from pytorch_tabular import TabularModel
from pytorch_tabular.models.ft_transformer.config import FTTransformerConfig
from pytorch_tabular.config import DataConfig, TrainerConfig, OptimizerConfig, ExperimentConfig

In [5]:
#ham veriyi ekleriz
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print(train_df.shape, test_df.shape)
train_df.head()

(1460, 81) (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [8]:
#hedef değişkeni ayırırız bu sayede regresyon hedefi netleşir.
target_col = "SalePrice"


In [9]:
#ıd sutununu ayırırız ama submisson için saklarız.

test_ids = test_df["Id"].copy()

train_df = train_df.drop(columns=["Id"])
test_df = test_df.drop(columns=["Id"])

In [11]:
#train ve test’i birleştirip temel feature enginerring yaparız.(yaş,banyo,verenda vb.)
#birleşik veri sayesinde (trainle testi birleştirme) tutarlı preprocessing(ön işleme) elde edilir.
"""
eğitim ve test verileri birleştirilerek tüm tablo üzerinde toplu bir işlem yapılmasına olanak sağlanıyor. Ardından, mevcut
sütunlar kullanılarak evin toplam yaşı veya toplam banyo sayısı gibi daha anlamlı ve kapsamlı yeni özellikler (feature engineering)
 oluşturuluyor.
"""
all_df = pd.concat([train_df.drop(columns=[target_col]), test_df], axis=0).reset_index(drop=True)

all_df["HouseAge"] = 2010 - all_df["YearBuilt"]
all_df["RemodAge"] = 2010 - all_df["YearRemodAdd"]
all_df["GarageAge"] = 2010 - all_df["GarageYrBlt"]

all_df["TotalSF"] = (
    all_df["TotalBsmtSF"].fillna(0)
    + all_df["1stFlrSF"].fillna(0)
    + all_df["2ndFlrSF"].fillna(0)
)

all_df["TotalBath"] = (
    all_df["FullBath"].fillna(0)
    + 0.5 * all_df["HalfBath"].fillna(0)
    + all_df["BsmtFullBath"].fillna(0)
    + 0.5 * all_df["BsmtHalfBath"].fillna(0)
)

all_df["TotalPorchSF"] = (
    all_df["OpenPorchSF"].fillna(0)
    + all_df["EnclosedPorch"].fillna(0)
    + all_df["3SsnPorch"].fillna(0)
    + all_df["ScreenPorch"].fillna(0)
)


In [12]:
#yok anlamına gelen kategorikleri none ile sayısalları median/0 ile doldururuz. Bu sayede eğitim sırasında null hatalarını engelleriz

none_cols = [
    "Alley", "PoolQC", "Fence", "MiscFeature", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType"
]

for col in none_cols:
    if col in all_df.columns:
        all_df[col] = all_df[col].fillna("None")

numeric_cols_all = all_df.select_dtypes(include=["int64", "float64"]).columns
categorical_cols_all = all_df.select_dtypes(include=["object"]).columns

for col in numeric_cols_all:
    all_df[col] = all_df[col].fillna(all_df[col].median())

for col in categorical_cols_all:
    all_df[col] = all_df[col].fillna("None")

In [13]:
#Kalite/durum belirten satırları sıralı sayıya çeviririz bu sayede model sütünlardaki düzen ilişkisini daha iyi anlar.
#Hiyerarşik düzen sağlamayı daha iyi anlar model.
qual_map = {
    "None": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
}

ordinal_cols = [
    "ExterQual", "ExterCond", "BsmtQual", "BsmtCond",
    "HeatingQC", "KitchenQual", "FireplaceQu",
    "GarageQual", "GarageCond", "PoolQC"
]

for col in ordinal_cols:
    if col in all_df.columns:
        all_df[col] = all_df[col].map(qual_map).fillna(0).astype(int)


In [14]:

# Feature engineering  sonrası tekrar train ve test olarak ayırırız bu sayede eğitim ve tahmin setleri netleşir
X_full = all_df.iloc[:len(train_df)].copy()
X_test = all_df.iloc[len(train_df):].copy()

y_full = np.log1p(train_df[target_col].copy())

train_ready = X_full.copy()
train_ready[target_col] = y_full

print(train_ready.shape, X_test.shape)


(1460, 86) (1459, 85)


In [15]:
#fttransformera hangi sütünün kategorisel hangisinin sayısal olduğunu veririz.
#bu sayede embedding ve sayısal girişler daha doğru işlenir
categorical_cols = train_ready.select_dtypes(include=["object"]).columns.tolist()
categorical_cols = [c for c in categorical_cols if c != target_col]

continuous_cols = [c for c in train_ready.columns if c not in categorical_cols + [target_col]]

print("Categorical:", len(categorical_cols))
print("Continuous :", len(continuous_cols))

Categorical: 33
Continuous : 52


In [16]:
"""
train ve validation bölmesini yaparız
eğitim ve doğrulama veri seti oluştururuz.
ft-transformerin performansını daha adil ölçeriz
"""
train_data, valid_data = train_test_split(
    train_ready,
    test_size=0.2,
    random_state=42
)

print(train_data.shape, valid_data.shape)

#modeli eğitmek için %80, performansını kontrol etmek için ise %20(test_sizee) oranında iki ayrı gruba ayırıyoruz

(1168, 86) (292, 86)


In [17]:
#Dataconfig oluşturma
# PyTorch Tabular’a hedef, kategorik ve sayısal sütunları tanımlarız bu sayede veri akışı daha doğru yapılır.
data_config = DataConfig(
    target=[target_col],
    continuous_cols=continuous_cols,
    categorical_cols=categorical_cols,
)

In [27]:
#TrainerConfig oluşturma
#Epoch, batch size ve erken durdurma gibi benzeri eğitim ayarlarını tanımlarız.
trainer_config = TrainerConfig(
    auto_lr_find=False,
    batch_size=32,
    max_epochs=150,
    accelerator="auto",
    early_stopping="valid_loss",
    early_stopping_patience=10,
    checkpoints="valid_loss",
    load_best=True
)

In [28]:
# Öğrenme oranı ve ağırlık çürümesi gibi optimizasyon ayarlarını tanımladık bu sayede eğitim daha stabil olur.
optimizer_config = OptimizerConfig(
    optimizer="AdamW",
    optimizer_params={"weight_decay": 1e-5},
    lr_scheduler="ReduceLROnPlateau",
    lr_scheduler_params={"patience": 3, "factor": 0.5},
)


In [29]:
# ExperimentConfig oluşturduk. Sonuçlar düzenli tutulur.
experiment_config = ExperimentConfig(
    project_name="ames_housing_ft_transformer",
    run_name="ft_transformer_regression_v1",
    exp_watch="gradients",
    log_target="tensorboard"
)


In [30]:
"""
FTTransformerConfig oluşturduk. FT-Transformer mimarisinin ana hiperparametrelerini belirledik
PyTorch Tabular tarafında görev türü regression olarak verilebiliyor ve regresyonda varsayılan olarak MSE takibi yapılıyor;
ayrıca TabularModel bu config’leri merkezden yönetebiliyor.
"""
model_config = FTTransformerConfig(
    task="regression",
    learning_rate=1e-4,
    input_embed_dim=32,
    num_heads=8,
    num_attn_blocks=4,
    attn_dropout=0.1,
    ff_dropout=0.1,
)


In [31]:
#Tüm config’leri bir araya getirip model nesnesini kurarız. Eğitim için pipeline hazır olur
tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
    experiment_config=experiment_config,
)

In [32]:
# FT-Transformer train ve validation verisiyle eğitilir.
#Tabular ilişkileri attention mekanizmasıyla öğrenir.
tabular_model.fit(
    train=train_data,
    validation=valid_data,
)


INFO:lightning_fabric.utilities.seed:Seed set to 42
2026-04-03 11:59:28,125 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-04-03 11:59:28,139 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
/usr/local/lib/python3.12/dist-packages/pytorch_tabular/tabular_datamodule.py:388: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.8667643   0.07410996 -0.63154574 ... -0.8667643  -0.16110861
  1.48542135]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[:, self.config.continuous_cols] = self.scaler.fit_transform(
/usr/local/lib/python3.12/dist-packages/pytorch_tabular/tabular_datamodule.py:388: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.21289571 -0.26524463 -0.17784146 ... -0.23409563 -0.28337613
 -0.65139925]' has dtyp

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ FTTransformerBackbone │  181 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding2dLayer      │ 12.7 K │ train │     0 │
│ 2 │ _head            │ LinearHead            │     33 │ train │     0 │
│ 3 │ loss             │ MSELoss               │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 193 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 193 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 152                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches 
(37) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if
you want to see logs for the training epoch.

2026-04-03 12:01:11,807 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-04-03 12:01:11,807 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


In [33]:
#Doğrulama verisi üzerinde tahmin üretiriz lineer regression ve catboost ile kıyasalamak için
pred_df = tabular_model.predict(valid_data)

pred_df.head()


#PyTorch Tabular sürümüne göre tahmin sütunu adı genelde prediction olur. Emin olmak için önce:

print(pred_df.columns)


#Sonra doğru sütunu al:

pred_col = "prediction" if "prediction" in pred_df.columns else pred_df.columns[-1]

y_true = valid_data[target_col].values
y_pred = pred_df[pred_col].values


/usr/local/lib/python3.12/dist-packages/pytorch_tabular/tabular_datamodule.py:392: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.8667643   0.07410996 -0.63154574 -0.16110861 -0.8667643   2.89673275
 -0.16110861  2.42629562  2.89673275 -0.16110861 -0.16110861 -0.8667643
 -0.63154574 -0.8667643   0.07410996 -0.8667643   0.07410996 -0.8667643
 -0.63154574 -0.8667643  -0.8667643   0.07410996  0.07410996 -0.8667643
 -0.8667643   1.48542135 -0.8667643  -0.8667643   0.07410996  1.48542135
 -0.16110861  0.07410996 -0.16110861 -0.8667643  -0.8667643  -0.8667643
 -0.16110861  0.07410996  0.07410996 -0.8667643  -0.16110861  0.07410996
 -0.8667643  -0.8667643  -0.8667643   0.30932853 -0.8667643  -0.8667643
  0.07410996 -0.16110861 -0.8667643  -0.8667643   2.42629562 -0.8667643
  1.48542135 -0.8667643   0.07410996 -0.8667643   0.54454709 -0.8667643
 -0.63154574 -0.8667643   0.07410996 -0.8667643  -0.8667643  -0.8667643
 -0.

Index(['SalePrice_prediction'], dtype='object')


In [34]:
#Metrikleri hesaplarız
mae_ft = mean_absolute_error(y_true, y_pred)
rmse_ft = np.sqrt(mean_squared_error(y_true, y_pred))
r2_ft = r2_score(y_true, y_pred)

print("FT-Transformer Validation Results")
print(f"MAE  : {mae_ft:.4f}")
print(f"RMSE : {rmse_ft:.4f}")
print(f"R2   : {r2_ft:.4f}")

FT-Transformer Validation Results
MAE  : 0.3332
RMSE : 0.4317
R2   : 0.0013


In [35]:
#test tahmin ve submissionu oluşturma
test_pred_df = tabular_model.predict(X_test)
pred_col_test = "prediction" if "prediction" in test_pred_df.columns else test_pred_df.columns[-1]

test_preds_log = test_pred_df[pred_col_test].values
test_preds = np.expm1(test_preds_log)

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": test_preds
})

submission.to_csv("submission_ft_transformer.csv", index=False)
submission.head()


/usr/local/lib/python3.12/dist-packages/pytorch_tabular/tabular_datamodule.py:392: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.8667643  -0.8667643   0.07410996 ... -0.8667643   0.66215637
  0.07410996]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[:, self.config.continuous_cols] = self.scaler.transform(data.loc[:, self.config.continuous_cols])
/usr/local/lib/python3.12/dist-packages/pytorch_tabular/tabular_datamodule.py:392: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 0.08669258  0.33263021  0.29199704 ...  0.86569654 -0.02311926
 -0.09880669]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[:, self.config.continuous_cols] = self.scaler.transform(data.loc[:, self.config.continuous_cols])
/usr/local/lib/python3.12/dist-packages/pytorc

,Id,SalePrice
0,1461,162771.953125
1,1462,162567.796875
2,1463,162816.359375
3,1464,162810.140625
4,1465,162767.609375
